### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex import Quinex
from quinex_application.visualisation_metadata import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [ ]:
quinex = Quinex()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2023")

### **Create Bulk Analysis**

In [ ]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

### **Import and Sort Paper by Year**

In [ ]:
visualisation.load_df( query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df

In [ ]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2023 = dfs_by_year[2023]
print(len(df_2023))

### **Add Papers via DOI**

In [ ]:
mongodb.insert_paper(name, df_2023)

#### **Finish remaining Papers**

In [ ]:
# Hole die DOIs (_id) aus beiden Collections als Sets
papers_2023_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2023")}
failed_papers_ids = {doc["_id"] for doc in mongodb.get_collection("failed_papers")}

# Vereine beide Sets (alle DOIs, die ausgeschlossen werden sollen)
excluded_ids = papers_2023_ids.union(failed_papers_ids)

# Filtere das DataFrame: nur Zeilen mit DOIs, die NICHT in excluded_ids enthalten sind
filtered_df = df_2023[~df_2023["DOI"].isin(excluded_ids)]
print(len(filtered_df))

In [ ]:
mongodb.insert_paper(name, filtered_df)

### **Successfully Added Papers**

In [ ]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_2023")
print(len(dois_df))

### **Perform Bulk Analysis**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=8)

### **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)

In [ ]:
analysis = mongodb.get_collection("papers_2024")
analysis[0]

### **Update Paper Parquet**

In [ ]:
df = mongodb.update_paper_parquet("papers_2024", "../data/paper/paper_dataset2.parquet", "03.06.2025", "17.04.2025")

In [ ]:
display(df[(df["Quinex"] == True)].head())